# Image QC
This notebook shows how image QC was handled. The idea is to use image-level
features to calculate a PCA on which we can do outlier detection. This is done
using functions of scmorph. The output is a distance of each
image's features to its kNN-nearest neighbor, which we use to score outlierness.
The distribution of these distances is the visualised to show where we may want
to set a cutoff for quality.

If you have not yet downloaded the required data from the CellPaintingGallery,
make sure to follow the instructions in the [README.md](README.md).

For a simpler introduction to image QC with scmorph, see: https://scmorph.readthedocs.io/en/latest/tutorials/basics/image_qc.html

In [1]:
import os
import scmorph as sm
from pathlib import Path
from pipeline_utils import (
    add_platemap,
    process_metadata,
    read_adata,
    read_miRNA_qc,
    read_platemap,
    sort_adata,
)

In [2]:
organisms = {
        "C2C12": "mouse",
        "HEK293T": "human",
        "HepG2": "human",
        "N2a": "mouse",
        "SHSY5Y": "human",
    }

# helper function that takes a single-cell feature file,
# an image-level feature file, the platemap file, and an imageQC
# threshold and performs all processing except for batch correction
# and feature selection
def preprocess(adata_file, qc_file, platemap, imageQC_threshold=0.1):
    cellline = os.path.basename(adata_file).split("_")[0]
    organism = organisms[cellline]

    pmap = read_platemap(platemap, organism)
    qc_adata = read_miRNA_qc(qc_file, cellline=cellline)
    adata = read_adata(adata_file, backed=False)
    adata = process_metadata(adata, cellline)
    adata = add_platemap(adata, pmap)
    adata, qc_adata = sm.qc.qc_images_by_dissimilarity(adata, qc_adata, filter=True, threshold=imageQC_threshold)
    adata = sort_adata(adata)
    sm.pp.drop_na(adata)

    if "Replicate" in adata.obs.columns:
        r1 = adata[adata.obs["Replicate"] == 1]
        r2 = adata[adata.obs["Replicate"] == 2]
        return [r1, r2]
    return [adata]


In [ ]:
QC_DIR = Path("workspace/qc/")
PROFILES_DIR = Path("workspace/profiles_assembled/")
OUTPUT_DIR = PROFILES_DIR / "profiles_qc"
MIRNA_PLATEMAP = Path("platemap/mirna_plate_layout_merged.csv")

qc_files = {
    c: QC_DIR / f"{c}_features_imageQC.h5ad"
    for c in organisms.keys()
}

sc_files = {
    cell_line: PROFILES_DIR / "profiles_raw" / f"{cell_line}_features.h5ad"
    for cell_line in organisms.keys()
}

for f in qc_files.values():
    assert f.exists, f"{f} does not exist"

for f in sc_files.values():
    assert f.exists, f"{f} does not exist"


Next, we will read in the features and perform QC using scmorph. This happens
using the helper function we declared further above. Since this process reads in
files of 20-40Gb and performs processing in-memory, it will not be very fast.

In [ ]:
for cell_line in organisms.keys():
    print(f"Processing {cell_line}...")
    adatas = preprocess(sc_files[cell_line], qc_files[cell_line], MIRNA_PLATEMAP)
    for repl, adata in enumerate(adatas):
        adata.write(OUTPUT_DIR / f"{cell_line}_R{repl+1}_features.h5ad")

Processing C2C12...


/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:
/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:


Processing HEK293T...


/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:
/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:


Processing HepG2...


/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:
/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:


Processing N2a...


/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:
/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:


Processing SHSY5Y...


/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:
/gpfs/igmmfs01/eddie/khamseh-lab/jwagner/projects/microrna-reproducibility/pipeline_utils.py:69: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not adata[0].obs["SampleName"].str.startswith("E")[0]:
